# Component 2 - Consensus Edge

Bets when one book's price beats the fair price implied by all the books together, using `scripts/consensus_edge.py`. **No stats model**: the books' average no-vig probability is the model. Needs the historical odds (`fetch_historical_odds.py`) and the game logs (*Player Stats Ingestion*) to settle bets. Takes under a minute.

**edge** = consensus chance of winning × best decimal odds − 1: the expected profit per $1 at the best price if the consensus is right. A player who didn't play voids the bet.

**Rules, fixed before looking at this season:**
- **Rule:** main line, 4+ books quoting, edge > 2%.
- **Same, Novig not the best price.**

They were found on last season's games (Jan-Apr). **This season's games are the real test**; don't change the rules based on them. Rerun after pulling more odds to grow the test.

Each row of the saved file is one side of one line: `consensus_p_win` and `fair_odds` (the books' no-vig view), `avg_odds`, `best_odds` / `best_book`, `edge`, `actual`, `outcome`, `profit_per_1`, and the `rule` / `rule_no_novig` flags.

In [ ]:
LEAGUE = "nba"

In [ ]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "consensus_edge.py").exists())
sys.path.insert(0, str(root / "scripts"))

import importlib
import fetch_player_stats, line_model, consensus_edge as ce
for m in (fetch_player_stats, line_model, ce):
    importlib.reload(m)  # pick up script edits without restarting the kernel

import pandas as pd
pd.set_option("display.max_columns", None)

## Build every bet

In [ ]:
bets = ce.run(LEAGUE)

## Did the rules hold up?

$1 flat on every rule bet at the best price. The 95% range resamples whole game dates.

In [ ]:
ce.rule_results(bets).round(3)

Does a bigger gap from the consensus mean a better return? Main lines with 4+ books:

In [ ]:
for sample in ("Last season (rule found here)", "This season (test)"):
    print(sample)
    display(ce.edge_buckets(bets, sample).round(3))

## This season's rule bets, broken down

In [ ]:
for col in ("best_book", "market", "side"):
    display(ce.by_group(bets, col).round(3))

In [ ]:
ce.daily(bets).round(2)

## Today's-style list

The rule bets on the latest date, biggest edge first.

In [ ]:
cols = ["game_date", "player", "team", "opponent", "market", "line", "side", "books", "consensus_p_win",
        "fair_odds", "avg_odds", "best_odds", "best_book", "edge", "actual", "outcome"]
last = bets.loc[bets["rule"], "game_date"].max()
bets.loc[bets["rule"] & (bets["game_date"] == last), cols].sort_values("edge", ascending=False).round(3)

## Save

In [ ]:
ce.save(LEAGUE, bets)